In [1]:
import rasterio
import numpy as np
from pylab import *
%matplotlib inline
import pandas as pd
import osgeo_utils.gdal_merge
from sklearn.ensemble import RandomForestClassifier
import joblib

from preprocessing_aux import *

In [ ]:
# RGB and chm rasters are currently not stacked, except for site CS-3A
# This code allows to stack the two .tif so that RGB and chm are stored in separate bands
# the resulting .tif has 5 bands (RGB, errorband, chm)

##### ACTIVATE ONLY WHEN STACKING IS NECESSARY ######
#output_file_path = r'C:/Users/MABEB16/science/postdoc/results/projects/lichen_mapping/color_analysis/paper/results/ortho_classification/merge.tif'
#input_files_path = [r'C:/Users/MABEB16/science/postdoc/results/projects/lichen_mapping/classification/qgis_CS3A_hp/CS3A_hp_transparent_mosaic_group1.tif', 
#                    'C:/Users/MABEB16/science/postdoc/results/projects/lichen_mapping/classification/qgis_CS3A_hp/dem_and_derivatives/CS3A_hp_stratified_chmheight_clipped.tif']
#parameters = ['', '-o', output_file_path] + input_files_path + ['-separate', '-co', 'COMPRESS=LZW']
#osgeo_utils.gdal_merge.main(parameters)

In [ ]:
# load classifier
clf_dir = '/data/clf/'
clf_name = 'clf_run6_orthotest.pkl'
clf = joblib.load(clf_dir+clf_name)
run = clf_name.split('_', maxsplit=1)[1][:-4]

In [ ]:
# load the stacked .tif file that has RGB and chm info
site = 'CS3A'
#ortho_dir = 'C:/Users/MABEB16/science/postdoc/results/projects/lichen_mapping/classification/qgis_%s_hp/' % site
ortho_fname = 'data/%s_hp_transparent_mosaic_group1_test_classification_stack2.tif' % site
ortho = ortho_fname

ds = rasterio.open(ortho)

In [ ]:
# raster calculations - calculate all required features for the loaded .tif that are required by the classifier 
# in order to determine veg classes

# RGB
red = ds.read(1).astype(float)
green = ds.read(2).astype(float)
blue = ds.read(3).astype(float)

# CHM
chm = ds.read(5).astype(float)

# prep
np.seterr(divide = "ignore", invalid = "ignore")
initializer = np.empty(ds.shape, dtype = rasterio.float32)
check = np.logical_or(red>0,green>0,blue>0)

# red chromaticity
rc = initializer
rc = np.where(check, red / (red + green + blue),0)

# green chromaticity
gc = initializer
gc = np.where(check, green / (red + green + blue),0)

# blue chromaticity
bc = initializer
bc = np.where(check, blue / (red + green + blue),0)

# rc/gc 
rcDivgc = initializer
rcDivgc = np.where(check, rc/gc, 0)

# rc/gc 
rcPlusgc = initializer
rcPlusgc = np.where(check, rc+gc, 0)

# ExG
ExG = initializer
ExG = np.where(check, 2*green-red-blue, 0)

# brightness calculations, (brightness Y and L, and z_score of brightness)
green_norm = np.where(check, green/(green+red+blue), 0)
red_norm = np.where(check, red/(green+red+blue), 0)
blue_norm = np.where(check, blue/(green+red+blue), 0)

greenLin = np.array([sRGBtoLin(val, single_value=False) for val in green_norm])
redLin = np.array([sRGBtoLin(val, single_value=False) for val in red_norm])
blueLin = np.array([sRGBtoLin(val, single_value=False) for val in blue_norm])

Y = initializer
Y = np.where(check, 0.2126*redLin + 0.7152*greenLin + 0.0722*blueLin, 0)
L = np.where(check, [YtoLstar(val, single_value=False) for val in Y], 0)

# brightness z_score
Yz = np.where(check, [z_score(x_all=val) for val in Y], 0)
Lz = np.where(check, [z_score(x_all=val) for val in L], 0)

plt.imshow(gc)

In [ ]:
# information about necessary features and their order
clf.col_names

In [ ]:
rows, cols = red.shape
bands=len(clf.col_names)

# stack all the arrays with the different features required by the clf,
# Info about the necessary features and the order of the features be is given in the cell above (see clf.col_names)
test = np.stack((red, green, blue, rc, gc,  bc, chm, rcDivgc, rcPlusgc, L, Y, Yz, Lz), axis=-1)
test2 = np.reshape(test, [rows*cols,bands])
# transform into a pandas dataframe
df_test = pd.DataFrame(test2, dtype='float32')
df_test = df_test.replace(np.NaN, 0)
df_test = df_test.replace(np.inf, 0)
# synchronize the feature column names with the names in the X dataframe, ie. the learning df
df_test.columns = clf.col_names

# predict veg class based on the features given in df_test. Classifier = clf
y_pred = clf.predict(df_test)
classified = y_pred.reshape((rows,cols))

In [ ]:
# write the classified array back into a tif and save on disk

out_img = "data/classified_%s_%s_2_1.tif" % (site, run)

file_list = [classified]
out_meta = ds.meta.copy()
out_meta.update({"count": len(file_list),
                 "dtype": rasterio.float32})

with rasterio.open(out_img, 'w', **out_meta) as dest:
    for band_nr, src in enumerate(file_list, start=1):
        dest.write(src, band_nr)